In [2]:
import sys
sys.path.insert(1, '/home/bricej/MyPythonLibrary/StageM2_IVT/library/')
import domain
from domain import *
from climbas import *
import numpy as np
import geopandas as gpd
import xarray as xr
import pandas as pd
from netCDF4 import Dataset
import dask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

In [ ]:
iyear = 2000
fyear = 2019

# Data for the tab

In [8]:
#dhdt 
dhdt = pd.read_csv('/bettik/PROJECTS/pr-regional-climate/bricej/paper/dhdt_2000-2019_HIMAP_regions.csv')

#era5 variables
data_era5 = {}
variables = ['t2m','FLH','tcwv','ivt','vimd','tp','sf','rf', 'R']

for var in variables:
    data_var = pd.read_csv(f'/bettik/PROJECTS/pr-regional-climate/bricej/paper/{var}_era5_trends_HIMAP_regions_{iyear}-{fyear}.csv')
    data_var = data_var[["region", "slope", "pvalue"]]
    data_var["slope"] = data_var["slope"] * 10
    data_era5[var] = data_var

# Making the tab with all data to have the overview

In [18]:
# fonction pour arrondir et ajouter * si pvalue < 0.05
def format_trend(value, pvalue, ndigits=2):
    if pd.isna(value):
        return ""
    txt = f"{value:.{ndigits}f}"
    if pvalue < 0.05:
        txt += "*"
    return txt



# Formater dhdt pour avoir value +/- err
df_dh = dhdt.copy()
df_dh["dhdt_str"] = df_dh.apply(
    lambda x: f"{x['dhdt']:.3f} ± {x['err_dhdt']:.3f}",
    axis=1
)
df_final = df_dh[["region", "dhdt_str"]]



# On ajoute les variables era5 pour avoir le tableau final
# nombre de décimales par variable
decimals = {
    "t2m": 2,
    "tp": 1,
    "sf": 1,
    "rf": 1,
    "tcwv": 2,
    "ivt": 2,
    "vimd": 3,
    "FLH": 0,
    "R": 2
}

for var, df_var in data_era5.items():
    nd = decimals.get(var, 2)  # défaut = 2 décimales
    df_tmp = df_var.copy()
    
    df_tmp[var] = df_tmp.apply(
        lambda x: format_trend(x["slope"], x["pvalue"], nd),
        axis=1
    )
    
    df_tmp = df_tmp[["region", var]]
    
    # merge avec le tableau final
    df_final = df_final.merge(df_tmp, on="region", how="left")

In [19]:
df_final

,region,dhdt_str,t2m,FLH,tcwv,ivt,vimd,tp,sf,rf,R
0,Eastern Hindu Kush,-0.193 ± 0.035,0.45,-2,0.27*,-0.23,-0.044,3.0,1.7,1.2,0.65
1,Western Himalaya,-0.383 ± 0.043,0.09,2,0.35*,0.54,-0.212,8.8*,3.5,5.3*,0.08
2,Eastern Himalaya,-0.538 ± 0.051,0.46*,53*,0.24*,1.61,-0.036,0.7,-1.4,2.0,-0.66
3,Central Himalaya,-0.477 ± 0.040,0.20,31,0.28,1.88,-0.205,6.0,0.4,5.6,-0.24
4,Karakoram,-0.073 ± 0.030,0.18,3,0.12,0.81,-0.070,1.4,2.3,-0.9,2.62
5,Western Pamir,-0.138 ± 0.033,0.48*,25,0.07,-0.21,-0.036,-0.2,0.2,-0.4,0.82
6,Pamir Alay,-0.122 ± 0.038,0.36,21,0.08,-0.86,0.006,-1.0,0.2,-1.3,0.95
7,Northern/Western Tien Shan,-0.411 ± 0.049,0.07,26,-0.03,-0.26,0.067,-3.4,-0.2,-3.2,1.33
8,Dzhungarsky Alatau,-0.618 ± 0.079,0.21,35,-0.03,0.51,0.051,-1.9,-0.1,-1.8,1.25
9,Western Kunlun Shan,0.115 ± 0.029,0.30,26,0.07,0.91*,0.013,-0.3,-1.2*,0.8,-2.68
